In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import imageio
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import optimizers

In [2]:
df = pd.read_csv('full_df_cleaned_v3.csv')

In [3]:
df.head()

,Diagnostic,file,target_init,Patient Age,Patient Sex,Target,tarstr,N,D,G,C,A,H,M,O,filename
0,normal fundus,0_right.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]",69,Female,"[1, 0, 0, 0, 0, 0, 0, 0]",N,1,0,0,0,0,0,0,0,0_right_69_Female_N.jpg
1,normal fundus,1_right.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]",57,Male,"[1, 0, 0, 0, 0, 0, 0, 0]",N,1,0,0,0,0,0,0,0,1_right_57_Male_N.jpg
2,moderate non proliferative retinopathy,2_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",42,Male,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,2_right_42_Male_D.jpg
3,mild nonproliferative retinopathy,4_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",53,Male,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,4_right_53_Male_D.jpg
4,moderate non proliferative retinopathy,5_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",50,Female,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,5_right_50_Female_D.jpg


In [4]:
df_n = df[df['tarstr']=='N']

In [5]:
df_n.shape

(2786, 16)

In [6]:
df_c = df[df['tarstr']=='C']

In [7]:
df_c.shape

(260, 16)

In [8]:
df_N_C = pd.concat([df_n, df_c], ignore_index=True)

In [9]:
df_N_C.shape

(3046, 16)

In [10]:
IMAGE_PATH = '../projectmini/preprocessed_images2/'

In [11]:
df_N_C['filepath'] = IMAGE_PATH + df_N_C['filename']

In [12]:
import imageio
img_data = []
number_id_nofile = []

for i in range(len(df_N_C)):
  try:
    img_data.append(imageio.imread(df_N_C['filepath'][i]))
  except:
    number_id_nofile.append(df_N_C.index[i])
      

C:\Users\sruja\AppData\Local\Temp\ipykernel_10212\704354572.py:7: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img_data.append(imageio.imread(df_N_C['filepath'][i]))


In [13]:
df_N_C = df_N_C.drop(number_id_nofile)

In [14]:
df_N_C['tarstr'].value_counts()

tarstr
N    2786
C     260
Name: count, dtype: int64

In [15]:
img_data_array = np.array(img_data)

In [16]:
img_data_array.shape

(3046, 256, 256, 3)

In [17]:
X = img_data_array

In [18]:
y = df_N_C['C']

In [19]:
y.shape

(3046,)

In [20]:
y.value_counts(1)

C
0    0.914642
1    0.085358
Name: proportion, dtype: float64

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, stratify=y)

In [22]:
X_train = X_train / 255
X_test = X_test / 255

In [23]:
df_N_C['C'].value_counts()

C
0    2786
1     260
Name: count, dtype: int64

In [24]:
y_train.value_counts(1)

C
0    0.914634
1    0.085366
Name: proportion, dtype: float64

In [29]:
from tensorflow.keras.applications.vgg16 import VGG16

def load_model():
   model = VGG16(include_top=False, weights='imagenet',input_shape=(256, 256, 3),
   Conv2D(32, (3,3), input_shape=(256, 256, 3), activation='relu', padding='same'),
   MaxPool2D(pool_size=(2,2)),
   Conv2D(64, (3,3), activation='relu', padding='same'),
   MaxPool2D(pool_size=(2,2)),
   Conv2D(128, (3,3), activation='relu', padding='same'),
   MaxPool2D(pool_size=(3,3)),
   Conv2D(256, (3,3), activation='relu', padding='same'),
   MaxPool2D(pool_size=(3,3)),
   ### Flattening
   Flatten(),
   ### One fully connected
   Dense(120, activation='relu'),
   Dropout(rate=0.5),
   Dense(60, activation='relu'),
   Dropout(rate=0.5),
   Dense(1, activation='sigmoid')
                )
   adam_opt = optimizers.Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999)

   model.compile(loss='binary_crossentropy', 
         optimizer=adam_opt,
         metrics=['accuracy'])

   return model

SyntaxError: positional argument follows keyword argument (2003283923.py, line 21)